# 01 — Data EDA

Exploratory analysis of the four-modality training corpus: example counts, token-length
distributions, and the realized multi-task mixture. Reads the prepared subsamples in
`data/raw/` (run `scripts/prepare_data.py` first if missing).

In [ ]:
import sys, json
from pathlib import Path
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))
from src.data.sources import iter_local_examples, take
from src.data.tokenizer import MODALITIES, build_tokenizer

DATA = REPO / 'data' / 'raw'
print('modalities:', MODALITIES)
print('data dir:', DATA, '| present:', DATA.exists())

In [ ]:
# Example counts per modality
counts = {m: sum(1 for _ in open(DATA / f'{m}.jsonl')) for m in MODALITIES if (DATA / f'{m}.jsonl').exists()}
print(counts)
plt.figure(figsize=(6,3))
plt.bar(counts.keys(), counts.values(), color='tab:blue')
plt.ylabel('examples'); plt.title('Examples per modality (local subsample)'); plt.show()

In [ ]:
# Token-length distribution per modality (tokenized with the OLMoE BPE tokenizer)
tok = build_tokenizer()
lengths = {}
for m in counts:
    exs = take(iter_local_examples(DATA / f'{m}.jsonl'), 200)
    lengths[m] = [len(tok.encode(e.prompt + e.completion, add_special_tokens=False)) for e in exs]

plt.figure(figsize=(8,4))
for m, L in lengths.items():
    plt.hist(L, bins=30, alpha=0.5, label=f'{m} (median {sorted(L)[len(L)//2]})')
plt.xlabel('tokens per example'); plt.ylabel('count'); plt.legend()
plt.title('Token-length distribution by modality'); plt.xlim(0, 1000); plt.show()

In [ ]:
# Realized mixture vs configured weights
from src.data.mixture import MixtureSampler, normalize_weights
from src.data.pipeline import make_local_sources

WEIGHTS = {'language': 0.40, 'code': 0.25, 'math': 0.20, 'logic': 0.15}
sampler = MixtureSampler(make_local_sources(DATA, MODALITIES), WEIGHTS, seed=0)
take(iter(sampler), 800)
realized = sampler.realized_mixture()

import numpy as np
x = np.arange(len(MODALITIES)); w = 0.35
plt.figure(figsize=(7,4))
plt.bar(x - w/2, [normalize_weights(WEIGHTS)[m] for m in MODALITIES], w, label='configured')
plt.bar(x + w/2, [realized[m] for m in MODALITIES], w, label='realized')
plt.xticks(x, MODALITIES); plt.ylabel('proportion'); plt.legend()
plt.title('Multi-task mixture: configured vs realized'); plt.show()
print('realized:', {k: round(v,3) for k,v in realized.items()})

In [ ]:
# Peek at one formatted, prompt-masked example (math): question masked, answer trained
from src.data.format import encode_example, IGNORE_INDEX
ex = next(iter_local_examples(DATA / 'math.jsonl'))
enc = encode_example(tok, ex, max_len=256)
n_trained = sum(l != IGNORE_INDEX for l in enc['labels'])
print('prompt:', ex.prompt[:120])
print('completion:', ex.completion[:120])
print(f'total tokens {len(enc["input_ids"])}, trained (non-masked) {n_trained}, masked {len(enc["labels"])-n_trained}')